In [1]:
import polars as pl
from datetime import datetime

In [2]:
player_stats_raw = pl.read_csv("../csv/PlayerStatistics.csv")
player_stats_raw.schema

Schema([('firstName', String),
        ('lastName', String),
        ('personId', Int64),
        ('gameId', Int64),
        ('gameDate', String),
        ('playerteamCity', String),
        ('playerteamName', String),
        ('opponentteamCity', String),
        ('opponentteamName', String),
        ('gameType', String),
        ('gameLabel', String),
        ('gameSubLabel', String),
        ('seriesGameNumber', Float64),
        ('win', Int64),
        ('home', Int64),
        ('numMinutes', Float64),
        ('points', Float64),
        ('assists', Float64),
        ('blocks', Float64),
        ('steals', Float64),
        ('fieldGoalsAttempted', Float64),
        ('fieldGoalsMade', Float64),
        ('fieldGoalsPercentage', Float64),
        ('threePointersAttempted', Float64),
        ('threePointersMade', Float64),
        ('threePointersPercentage', Float64),
        ('freeThrowsAttempted', Float64),
        ('freeThrowsMade', Float64),
        ('freeThrowsPercentage', Float64

In [3]:
# Step 1: Parse the string column into Datetime
box_score = player_stats_raw.with_columns(
    pl.col("gameDate").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
)

# Step 2: Filter rows where gameDate >= 1980-07-16
cutoff_date = datetime(1980, 7, 16)
box_score = box_score.filter(pl.col("gameDate") >= cutoff_date).filter(pl.col("gameType") == "Regular Season")
player_stats = box_score.drop(["gameId","playerteamName", "opponentteamCity", "opponentteamName", "gameType", "gameLabel", "gameSubLabel", "seriesGameNumber", "win", "home", "fieldGoalsPercentage", "threePointersPercentage","freeThrowsPercentage"])
player_stats

firstName,lastName,personId,gameDate,playerteamCity,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,encodedTeam
str,str,i64,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""Jeff""","""Green""",201145,2025-04-13 15:30:00,"""Houston""",11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,2.0,1.0,3.0,1.0,0.0,-14.0,11
"""Russell""","""Westbrook""",201566,2025-04-13 15:30:00,"""Denver""",22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,0.0,0.0,1.0,1.0,8.0,8
"""DeAndre""","""Jordan""",201599,2025-04-13 15:30:00,"""Denver""",10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,3.0,2.0,5.0,0.0,0.0,-9.0,8
"""Steven""","""Adams""",203500,2025-04-13 15:30:00,"""Houston""",17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,4.0,2.0,6.0,2.0,1.0,-23.0,11
"""Aaron""","""Gordon""",203932,2025-04-13 15:30:00,"""Denver""",26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,3.0,4.0,7.0,0.0,3.0,21.0,8
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Louis""","""Orr""",77770,1980-10-10 20:00:00,"""Indiana""",8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,0.0,0.0,12
"""Cliff T.""","""Robinson""",77986,1980-10-10 20:00:00,"""New Jersey""",29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,0.0,0.0,9.0,3.0,6.0,0.0,19
"""Jerry""","""Sichting""",78146,1980-10-10 20:00:00,"""Indiana""",10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,12


In [4]:
player_stats = player_stats.with_columns(
    pl.when(pl.col("gameDate").dt.month() > 7)
      .then(pl.col("gameDate").dt.year())
      .otherwise(pl.col("gameDate").dt.year() - 1)
      .alias("season")
)
player_stats = player_stats.drop(["firstName", "lastName","gameDate"])
player_stats

personId,playerteamCity,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,encodedTeam,season
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i32
201145,"""Houston""",11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,2.0,1.0,3.0,1.0,0.0,-14.0,11,2024
201566,"""Denver""",22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,0.0,0.0,1.0,1.0,8.0,8,2024
201599,"""Denver""",10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,3.0,2.0,5.0,0.0,0.0,-9.0,8,2024
203500,"""Houston""",17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,4.0,2.0,6.0,2.0,1.0,-23.0,11,2024
203932,"""Denver""",26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,3.0,4.0,7.0,0.0,3.0,21.0,8,2024
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
77770,"""Indiana""",8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,0.0,0.0,12,1980
77986,"""New Jersey""",29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,0.0,0.0,9.0,3.0,6.0,0.0,19,1980
78146,"""Indiana""",10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,12,1980


In [5]:
# Get unique values and dense ranks
mapping_df = player_stats.select([
    pl.col("playerteamCity").unique().sort().alias("playerteamCity")
]).with_columns([
    pl.col("playerteamCity").rank("dense").cast(pl.Int64).alias("encodedTeam")
])

# Convert to a dictionary
mapping_dict = dict(zip(mapping_df["playerteamCity"].to_list(), mapping_df["encodedTeam"].to_list()))
print(mapping_dict)
player_stats = player_stats.with_columns(encodedTeam = pl.col("playerteamCity").rank("dense")).drop("playerteamCity")

{'Atlanta': 1, 'Boston': 2, 'Brooklyn': 3, 'Charlotte': 4, 'Chicago': 5, 'Cleveland': 6, 'Dallas': 7, 'Denver': 8, 'Detroit': 9, 'Golden State': 10, 'Houston': 11, 'Indiana': 12, 'Kansas City': 13, 'Los Angeles': 14, 'Memphis': 15, 'Miami': 16, 'Milwaukee': 17, 'Minnesota': 18, 'New Jersey': 19, 'New Orleans': 20, 'New York': 21, 'Oklahoma City': 22, 'Orlando': 23, 'Philadelphia': 24, 'Phoenix': 25, 'Portland': 26, 'Sacramento': 27, 'San Antonio': 28, 'San Diego': 29, 'Seattle': 30, 'Toronto': 31, 'Utah': 32, 'Vancouver': 33, 'Washington': 34}


In [6]:
player_stats

personId,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,encodedTeam,season
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32
201145,11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,2.0,1.0,3.0,1.0,0.0,-14.0,11,2024
201566,22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,0.0,0.0,1.0,1.0,8.0,8,2024
201599,10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,3.0,2.0,5.0,0.0,0.0,-9.0,8,2024
203500,17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,4.0,2.0,6.0,2.0,1.0,-23.0,11,2024
203932,26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,3.0,4.0,7.0,0.0,3.0,21.0,8,2024
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
77770,8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,0.0,0.0,12,1980
77986,29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,0.0,0.0,9.0,3.0,6.0,0.0,19,1980
78146,10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,12,1980


In [7]:
per_game = player_stats.group_by(["personId", "season", "encodedTeam"]).mean().drop(["reboundsDefensive", "reboundsOffensive", "foulsPersonal", "plusMinusPoints"])

In [8]:
per_game

personId,season,encodedTeam,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers
i64,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2555,2015,22,11.322034,1.5,0.634146,0.195122,0.182927,1.329268,0.609756,0.02439,0.0,0.402439,0.280488,2.085366,0.609756
201160,2007,24,14.078947,4.303797,0.253165,0.632911,0.278481,3.924051,1.78481,0.177215,0.050633,1.037975,0.683544,2.911392,0.607595
781,2000,26,3.769231,0.237288,0.033898,0.033898,0.050847,0.152542,0.101695,0.0,0.0,0.067797,0.033898,0.305085,0.0
1628385,2019,27,14.461538,4.684211,0.877193,0.245614,0.368421,3.666667,2.052632,0.0,0.0,0.754386,0.578947,2.807018,0.614035
2410,2003,12,17.950617,4.841463,2.109756,0.219512,0.792683,3.792683,1.5,1.085366,0.329268,1.817073,1.512195,1.536585,0.890244
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
201595,2009,27,6.125,0.923077,0.0,0.076923,0.076923,0.692308,0.307692,0.0,0.0,0.769231,0.307692,1.384615,0.153846
76441,1989,26,14.886076,3.810127,0.556962,1.202532,0.227848,3.848101,1.746835,0.037975,0.0,0.493671,0.316456,4.291139,0.493671
1626246,2021,7,5.130435,1.289474,0.039474,0.039474,0.0,0.921053,0.552632,0.052632,0.013158,0.289474,0.171053,0.513158,0.197368


In [9]:
player_stats

personId,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,encodedTeam,season
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32
201145,11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,2.0,1.0,3.0,1.0,0.0,-14.0,11,2024
201566,22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,0.0,0.0,1.0,1.0,8.0,8,2024
201599,10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,3.0,2.0,5.0,0.0,0.0,-9.0,8,2024
203500,17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,4.0,2.0,6.0,2.0,1.0,-23.0,11,2024
203932,26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,3.0,4.0,7.0,0.0,3.0,21.0,8,2024
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
77770,8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,0.0,0.0,12,1980
77986,29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,0.0,0.0,9.0,3.0,6.0,0.0,19,1980
78146,10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,12,1980


In [10]:
per_game.write_csv("../csv/PerGame.csv")

In [11]:
games = pl.read_csv("../csv/Games.csv")
games

gameId,gameDate,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,gameType,attendance,arenaId,gameLabel,gameSubLabel,seriesGameNumber
i64,str,str,str,i64,str,str,i64,i64,i64,i64,str,f64,i64,str,str,f64
42400407,"""2025-06-22 20:00:00""","""Oklahoma City""","""Thunder""",1610612760,"""Indiana""","""Pacers""",1610612754,103,91,1610612760,"""Playoffs""",18203.0,1000052,"""NBA Finals""","""Game 7""",7.0
42400406,"""2025-06-19 20:30:00""","""Indiana""","""Pacers""",1610612754,"""Oklahoma City""","""Thunder""",1610612760,108,91,1610612754,"""Playoffs""",17274.0,1000063,"""NBA Finals""","""Game 6""",6.0
42400405,"""2025-06-16 20:30:00""","""Oklahoma City""","""Thunder""",1610612760,"""Indiana""","""Pacers""",1610612754,120,109,1610612760,"""Playoffs""",18203.0,1000052,"""NBA Finals""","""Game 5""",5.0
42400404,"""2025-06-13 20:30:00""","""Indiana""","""Pacers""",1610612754,"""Oklahoma City""","""Thunder""",1610612760,104,111,1610612760,"""Playoffs""",17274.0,1000063,"""NBA Finals""","""Game 4""",4.0
42400403,"""2025-06-11 20:30:00""","""Indiana""","""Pacers""",1610612754,"""Oklahoma City""","""Thunder""",1610612760,116,107,1610612754,"""Playoffs""",17274.0,1000063,"""NBA Finals""","""Game 3""",3.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
24600083,"""1946-12-08 19:00:00""","""New York""","""Knicks""",1610612752,"""Boston""","""Celtics""",1610612738,62,44,1610612752,"""Regular Season""",null,0,null,null,null
24600078,"""1946-12-07 19:00:00""","""Boston""","""Celtics""",1610612738,"""New York""","""Knicks""",1610612752,65,90,1610612752,"""Regular Season""",null,0,null,null,null
24600076,"""1946-12-05 19:00:00""","""Philadelphia""","""Warriors""",1610612744,"""New York""","""Knicks""",1610612752,62,51,1610612744,"""Regular Season""",null,0,null,null,null


In [12]:
games = games.with_columns(
    pl.col("hometeamCity").map_elements(lambda x: mapping_dict.get(x, -1), return_dtype=pl.Int64).alias("encodedHomeTeam")
)

games = games.with_columns(
    pl.col("awayteamCity").map_elements(lambda x: mapping_dict.get(x, -1), return_dtype=pl.Int64).alias("encodedAwayTeam")
)

games

gameId,gameDate,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,gameType,attendance,arenaId,gameLabel,gameSubLabel,seriesGameNumber,encodedHomeTeam,encodedAwayTeam
i64,str,str,str,i64,str,str,i64,i64,i64,i64,str,f64,i64,str,str,f64,i64,i64
42400407,"""2025-06-22 20:00:00""","""Oklahoma City""","""Thunder""",1610612760,"""Indiana""","""Pacers""",1610612754,103,91,1610612760,"""Playoffs""",18203.0,1000052,"""NBA Finals""","""Game 7""",7.0,22,12
42400406,"""2025-06-19 20:30:00""","""Indiana""","""Pacers""",1610612754,"""Oklahoma City""","""Thunder""",1610612760,108,91,1610612754,"""Playoffs""",17274.0,1000063,"""NBA Finals""","""Game 6""",6.0,12,22
42400405,"""2025-06-16 20:30:00""","""Oklahoma City""","""Thunder""",1610612760,"""Indiana""","""Pacers""",1610612754,120,109,1610612760,"""Playoffs""",18203.0,1000052,"""NBA Finals""","""Game 5""",5.0,22,12
42400404,"""2025-06-13 20:30:00""","""Indiana""","""Pacers""",1610612754,"""Oklahoma City""","""Thunder""",1610612760,104,111,1610612760,"""Playoffs""",17274.0,1000063,"""NBA Finals""","""Game 4""",4.0,12,22
42400403,"""2025-06-11 20:30:00""","""Indiana""","""Pacers""",1610612754,"""Oklahoma City""","""Thunder""",1610612760,116,107,1610612754,"""Playoffs""",17274.0,1000063,"""NBA Finals""","""Game 3""",3.0,12,22
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
24600083,"""1946-12-08 19:00:00""","""New York""","""Knicks""",1610612752,"""Boston""","""Celtics""",1610612738,62,44,1610612752,"""Regular Season""",null,0,null,null,null,21,2
24600078,"""1946-12-07 19:00:00""","""Boston""","""Celtics""",1610612738,"""New York""","""Knicks""",1610612752,65,90,1610612752,"""Regular Season""",null,0,null,null,null,2,21
24600076,"""1946-12-05 19:00:00""","""Philadelphia""","""Warriors""",1610612744,"""New York""","""Knicks""",1610612752,62,51,1610612744,"""Regular Season""",null,0,null,null,null,24,21


In [13]:
# Step 1: Parse the string column into Datetime
games = games.with_columns(
    pl.col("gameDate").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
)

# Step 2: Filter rows where gameDate >= 1980-07-16
cutoff_date = datetime(1980, 7, 16)
games = games.filter(pl.col("gameDate") >= cutoff_date).filter(pl.col("gameType") == "Regular Season").drop(["hometeamCity", "hometeamName", "hometeamId", "awayteamCity", "awayteamName", "awayteamId", "winner", "gameType", "attendance","arenaId", "gameLabel", "gameSubLabel", "seriesGameNumber"])
games

gameId,gameDate,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam
i64,datetime[μs],i64,i64,i64,i64
22401193,2025-04-13 15:30:00,111,126,11,8
22401194,2025-04-13 15:30:00,132,97,15,7
22401195,2025-04-13 15:30:00,116,105,18,32
22401196,2025-04-13 15:30:00,100,115,20,22
22401197,2025-04-13 15:30:00,125,118,28,31
…,…,…,…,…,…
28000005,1980-10-10 20:00:00,85,95,9,34
28000006,1980-10-10 20:00:00,98,99,30,14
28000007,1980-10-10 20:00:00,130,103,2,6


In [14]:
games = games.with_columns(
    pl.when(pl.col("gameDate").dt.month() > 7)
      .then(pl.col("gameDate").dt.year())
      .otherwise(pl.col("gameDate").dt.year() - 1)
      .alias("season")
).drop("gameDate")

games


gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season
i64,i64,i64,i64,i64,i32
22401193,111,126,11,8,2024
22401194,132,97,15,7,2024
22401195,116,105,18,32,2024
22401196,100,115,20,22,2024
22401197,125,118,28,31,2024
…,…,…,…,…,…
28000005,85,95,9,34,1980
28000006,98,99,30,14,1980
28000007,130,103,2,6,1980


In [15]:
per_game

personId,season,encodedTeam,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers
i64,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2555,2015,22,11.322034,1.5,0.634146,0.195122,0.182927,1.329268,0.609756,0.02439,0.0,0.402439,0.280488,2.085366,0.609756
201160,2007,24,14.078947,4.303797,0.253165,0.632911,0.278481,3.924051,1.78481,0.177215,0.050633,1.037975,0.683544,2.911392,0.607595
781,2000,26,3.769231,0.237288,0.033898,0.033898,0.050847,0.152542,0.101695,0.0,0.0,0.067797,0.033898,0.305085,0.0
1628385,2019,27,14.461538,4.684211,0.877193,0.245614,0.368421,3.666667,2.052632,0.0,0.0,0.754386,0.578947,2.807018,0.614035
2410,2003,12,17.950617,4.841463,2.109756,0.219512,0.792683,3.792683,1.5,1.085366,0.329268,1.817073,1.512195,1.536585,0.890244
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
201595,2009,27,6.125,0.923077,0.0,0.076923,0.076923,0.692308,0.307692,0.0,0.0,0.769231,0.307692,1.384615,0.153846
76441,1989,26,14.886076,3.810127,0.556962,1.202532,0.227848,3.848101,1.746835,0.037975,0.0,0.493671,0.316456,4.291139,0.493671
1626246,2021,7,5.130435,1.289474,0.039474,0.039474,0.0,0.921053,0.552632,0.052632,0.013158,0.289474,0.171053,0.513158,0.197368


In [16]:
# box_score = box_score.drop(["playerteamCity", "playerteamName",	"opponentteamCity",	"opponentteamName",	"gameType",	"gameLabel","gameSubLabel",	"seriesGameNumber", "win", "home", "fieldGoalsPercentage", "threePointersPercentage", "freeThrowsPercentage", "reboundsDefensive", "reboundsOffensive", "foulsPersonal", "plusMinusPoints"])
box_score = box_score.drop([	"gameType",	"gameLabel","gameSubLabel",	"seriesGameNumber", "win", "home", "fieldGoalsPercentage", "threePointersPercentage", "freeThrowsPercentage", "reboundsDefensive", "reboundsOffensive", "foulsPersonal", "plusMinusPoints"])
box_score = box_score.with_columns(
    pl.when(pl.col("gameDate").dt.month() > 7)
      .then(pl.col("gameDate").dt.year())
      .otherwise(pl.col("gameDate").dt.year() - 1)
      .alias("season")
).drop("gameDate")

In [17]:
box_score

firstName,lastName,personId,gameId,playerteamCity,playerteamName,opponentteamCity,opponentteamName,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeam,season
str,str,i64,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i32
"""Jeff""","""Green""",201145,22401193,"""Houston""","""Rockets""","""Denver""","""Nuggets""",11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,3.0,0.0,11,2024
"""Russell""","""Westbrook""",201566,22401193,"""Denver""","""Nuggets""","""Houston""","""Rockets""",22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,1.0,8,2024
"""DeAndre""","""Jordan""",201599,22401193,"""Denver""","""Nuggets""","""Houston""","""Rockets""",10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,5.0,0.0,8,2024
"""Steven""","""Adams""",203500,22401193,"""Houston""","""Rockets""","""Denver""","""Nuggets""",17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,6.0,1.0,11,2024
"""Aaron""","""Gordon""",203932,22401193,"""Denver""","""Nuggets""","""Houston""","""Rockets""",26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,7.0,3.0,8,2024
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Louis""","""Orr""",77770,28000009,"""Indiana""","""Pacers""","""New Jersey""","""Nets""",8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,2.0,0.0,12,1980
"""Cliff T.""","""Robinson""",77986,28000009,"""New Jersey""","""Nets""","""Indiana""","""Pacers""",29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,9.0,6.0,19,1980
"""Jerry""","""Sichting""",78146,28000009,"""Indiana""","""Pacers""","""New Jersey""","""Nets""",10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,12,1980


In [18]:
box_score_with_per_game = box_score.join(per_game, on=["personId", "season"], how="left", suffix="pg")
box_score_with_per_game

firstName,lastName,personId,gameId,playerteamCity,playerteamName,opponentteamCity,opponentteamName,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeam,season,encodedTeampg,numMinutespg,pointspg,assistspg,blockspg,stealspg,fieldGoalsAttemptedpg,fieldGoalsMadepg,threePointersAttemptedpg,threePointersMadepg,freeThrowsAttemptedpg,freeThrowsMadepg,reboundsTotalpg,turnoverspg
str,str,i64,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Jeff""","""Green""",201145,22401193,"""Houston""","""Rockets""","""Denver""","""Nuggets""",11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,3.0,0.0,11,2024,11,12.183438,2.123457,0.246914,0.049383,0.074074,1.493827,0.753086,0.975309,0.358025,0.320988,0.259259,0.716049,0.111111
"""Russell""","""Westbrook""",201566,22401193,"""Denver""","""Nuggets""","""Houston""","""Rockets""",22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,1.0,8,2024,8,27.6936,13.253333,6.093333,0.493333,1.413333,11.08,4.973333,3.88,1.253333,3.106667,2.053333,4.933333,3.226667
"""DeAndre""","""Jordan""",201599,22401193,"""Denver""","""Nuggets""","""Houston""","""Rockets""",10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,5.0,0.0,8,2024,8,12.143036,2.54878,0.646341,0.329268,0.195122,1.707317,1.109756,0.0,0.0,0.780488,0.329268,3.463415,0.487805
"""Steven""","""Adams""",203500,22401193,"""Houston""","""Rockets""","""Denver""","""Nuggets""",17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,6.0,1.0,11,2024,11,13.483621,3.214286,0.942857,0.4,0.314286,2.385714,1.3,0.028571,0.0,1.328571,0.614286,4.671429,0.771429
"""Aaron""","""Gordon""",203932,22401193,"""Denver""","""Nuggets""","""Houston""","""Rockets""",26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,7.0,3.0,8,2024,8,28.173137,14.666667,3.215686,0.27451,0.45098,9.745098,5.176471,3.372549,1.470588,3.509804,2.843137,4.843137,1.431373
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Louis""","""Orr""",77770,28000009,"""Indiana""","""Pacers""","""New Jersey""","""Nets""",8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,2.0,0.0,12,1980,12,21.792683,10.47561,1.609756,0.219512,0.292683,8.536585,4.243902,0.073171,0.0,2.463415,1.987805,4.402439,0.756098
"""Cliff T.""","""Robinson""",77986,28000009,"""New Jersey""","""Nets""","""Indiana""","""Pacers""",29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,9.0,6.0,19,1980,19,28.888889,19.507937,1.603175,0.52381,0.285714,16.984127,8.333333,0.015873,0.015873,3.936508,2.825397,7.603175,1.31746
"""Jerry""","""Sichting""",78146,28000009,"""Indiana""","""Pacers""","""New Jersey""","""Nets""",10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,12,1980,12,9.574468,1.978723,1.489362,0.021277,0.191489,1.93617,0.723404,0.06383,0.0,0.680851,0.531915,0.914894,0.191489


In [19]:
# pivot_df = box_score_with_per_game.pivot(
# values=[
#         "numMinutespg", "pointspg", "assistspg", "blockspg", "stealspg",
#         "fieldGoalsAttemptedpg", "fieldGoalsMadepg", "threePointersAttemptedpg",
#         "threePointersMadepg", "freeThrowsAttemptedpg", "freeThrowsMadepg",
#         "reboundsDefensive", "reboundsOffensive", "reboundsTotalpg",
#         "foulsPersonal", "turnoverspg", "plusMinusPoints"
#     ],    
#     index="gameId",
#     on="personId",
#     aggregate_function="first"             # assuming one stat row per game/player
# )

# pivot_df

In [20]:
master = games.join(box_score_with_per_game, on=["gameId", "season"], how="left")
master

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season,firstName,lastName,personId,playerteamCity,playerteamName,opponentteamCity,opponentteamName,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeam,encodedTeampg,numMinutespg,pointspg,assistspg,blockspg,stealspg,fieldGoalsAttemptedpg,fieldGoalsMadepg,threePointersAttemptedpg,threePointersMadepg,freeThrowsAttemptedpg,freeThrowsMadepg,reboundsTotalpg,turnoverspg
i64,i64,i64,i64,i64,i32,str,str,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
22401193,111,126,11,8,2024,"""Jeff""","""Green""",201145,"""Houston""","""Rockets""","""Denver""","""Nuggets""",11.31,5.0,0.0,0.0,0.0,6.0,2.0,5.0,1.0,0.0,0.0,3.0,0.0,11,11,12.183438,2.123457,0.246914,0.049383,0.074074,1.493827,0.753086,0.975309,0.358025,0.320988,0.259259,0.716049,0.111111
22401193,111,126,11,8,2024,"""Russell""","""Westbrook""",201566,"""Denver""","""Nuggets""","""Houston""","""Rockets""",22.39,17.0,6.0,0.0,0.0,9.0,5.0,3.0,0.0,8.0,7.0,0.0,1.0,8,8,27.6936,13.253333,6.093333,0.493333,1.413333,11.08,4.973333,3.88,1.253333,3.106667,2.053333,4.933333,3.226667
22401193,111,126,11,8,2024,"""DeAndre""","""Jordan""",201599,"""Denver""","""Nuggets""","""Houston""","""Rockets""",10.52,4.0,0.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,5.0,0.0,8,8,12.143036,2.54878,0.646341,0.329268,0.195122,1.707317,1.109756,0.0,0.0,0.780488,0.329268,3.463415,0.487805
22401193,111,126,11,8,2024,"""Steven""","""Adams""",203500,"""Houston""","""Rockets""","""Denver""","""Nuggets""",17.13,4.0,1.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,6.0,1.0,11,11,13.483621,3.214286,0.942857,0.4,0.314286,2.385714,1.3,0.028571,0.0,1.328571,0.614286,4.671429,0.771429
22401193,111,126,11,8,2024,"""Aaron""","""Gordon""",203932,"""Denver""","""Nuggets""","""Houston""","""Rockets""",26.23,18.0,3.0,1.0,2.0,13.0,7.0,3.0,1.0,3.0,3.0,7.0,3.0,8,8,28.173137,14.666667,3.215686,0.27451,0.45098,9.745098,5.176471,3.372549,1.470588,3.509804,2.843137,4.843137,1.431373
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
28000009,91,110,19,12,1980,"""Louis""","""Orr""",77770,"""Indiana""","""Pacers""","""New Jersey""","""Nets""",8.0,4.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,2.0,2.0,0.0,12,12,21.792683,10.47561,1.609756,0.219512,0.292683,8.536585,4.243902,0.073171,0.0,2.463415,1.987805,4.402439,0.756098
28000009,91,110,19,12,1980,"""Cliff T.""","""Robinson""",77986,"""New Jersey""","""Nets""","""Indiana""","""Pacers""",29.0,11.0,3.0,2.0,0.0,15.0,5.0,0.0,0.0,2.0,1.0,9.0,6.0,19,19,28.888889,19.507937,1.603175,0.52381,0.285714,16.984127,8.333333,0.015873,0.015873,3.936508,2.825397,7.603175,1.31746
28000009,91,110,19,12,1980,"""Jerry""","""Sichting""",78146,"""Indiana""","""Pacers""","""New Jersey""","""Nets""",10.0,4.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,12,12,9.574468,1.978723,1.489362,0.021277,0.191489,1.93617,0.723404,0.06383,0.0,0.680851,0.531915,0.914894,0.191489


In [21]:
unique_game_ids = games.select(pl.col("gameId").unique()).sort(by="gameId")
unique_game_ids

gameId
i64
20000001
20000002
20000003
20000004
20000005
…
29901186
29901187
29901188


In [22]:
games

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season
i64,i64,i64,i64,i64,i32
22401193,111,126,11,8,2024
22401194,132,97,15,7,2024
22401195,116,105,18,32,2024
22401196,100,115,20,22,2024
22401197,125,118,28,31,2024
…,…,…,…,…,…
28000005,85,95,9,34,1980
28000006,98,99,30,14,1980
28000007,130,103,2,6,1980


In [23]:

new_columns = [
    "numMinutes",
    "points",
    "assists",
    "blocks",
    "steals",
    "reboundsTotal",
    "turnovers",
]


cols = []

for teamIdx in range(2):
    teamStr = 't' + str(teamIdx)
    for playerIdx in range(15):
        playerStr = 'p' + str(playerIdx)
        prefix = teamStr + '_' + playerStr + '_'
        for col in new_columns:
            cols.append(prefix + col)
        
print(cols)
master = games.with_columns([
    pl.lit(0).alias(col_name) for col_name in cols
])
new_columns = [
    "numMinutespg",
    "pointspg",
    "assistspg",
    "blockspg",
    "stealspg",
    "fieldGoalsAttemptedpg",
    "fieldGoalsMadepg",
    "threePointersAttemptedpg",
    "threePointersMadepg",
    "freeThrowsAttemptedpg",
    "freeThrowsMadepg",
    "reboundsTotalpg",
    "turnoverspg",
]
cols = []

for teamIdx in range(2):
    teamStr = 't' + str(teamIdx)
    for playerIdx in range(15):
        playerStr = 'p' + str(playerIdx)
        prefix = teamStr + '_' + playerStr + '_'
        for col in new_columns:
            cols.append(prefix + col)
        
print(cols)
master = master.with_columns([
    pl.lit(0).alias(col_name) for col_name in cols
])

master

['t0_p0_numMinutes', 't0_p0_points', 't0_p0_assists', 't0_p0_blocks', 't0_p0_steals', 't0_p0_reboundsTotal', 't0_p0_turnovers', 't0_p1_numMinutes', 't0_p1_points', 't0_p1_assists', 't0_p1_blocks', 't0_p1_steals', 't0_p1_reboundsTotal', 't0_p1_turnovers', 't0_p2_numMinutes', 't0_p2_points', 't0_p2_assists', 't0_p2_blocks', 't0_p2_steals', 't0_p2_reboundsTotal', 't0_p2_turnovers', 't0_p3_numMinutes', 't0_p3_points', 't0_p3_assists', 't0_p3_blocks', 't0_p3_steals', 't0_p3_reboundsTotal', 't0_p3_turnovers', 't0_p4_numMinutes', 't0_p4_points', 't0_p4_assists', 't0_p4_blocks', 't0_p4_steals', 't0_p4_reboundsTotal', 't0_p4_turnovers', 't0_p5_numMinutes', 't0_p5_points', 't0_p5_assists', 't0_p5_blocks', 't0_p5_steals', 't0_p5_reboundsTotal', 't0_p5_turnovers', 't0_p6_numMinutes', 't0_p6_points', 't0_p6_assists', 't0_p6_blocks', 't0_p6_steals', 't0_p6_reboundsTotal', 't0_p6_turnovers', 't0_p7_numMinutes', 't0_p7_points', 't0_p7_assists', 't0_p7_blocks', 't0_p7_steals', 't0_p7_reboundsTotal', 't

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season,t0_p0_numMinutes,t0_p0_points,t0_p0_assists,t0_p0_blocks,t0_p0_steals,t0_p0_reboundsTotal,t0_p0_turnovers,t0_p1_numMinutes,t0_p1_points,t0_p1_assists,t0_p1_blocks,t0_p1_steals,t0_p1_reboundsTotal,t0_p1_turnovers,t0_p2_numMinutes,t0_p2_points,t0_p2_assists,t0_p2_blocks,t0_p2_steals,t0_p2_reboundsTotal,t0_p2_turnovers,t0_p3_numMinutes,t0_p3_points,t0_p3_assists,t0_p3_blocks,t0_p3_steals,t0_p3_reboundsTotal,t0_p3_turnovers,t0_p4_numMinutes,t0_p4_points,t0_p4_assists,…,t1_p12_assistspg,t1_p12_blockspg,t1_p12_stealspg,t1_p12_fieldGoalsAttemptedpg,t1_p12_fieldGoalsMadepg,t1_p12_threePointersAttemptedpg,t1_p12_threePointersMadepg,t1_p12_freeThrowsAttemptedpg,t1_p12_freeThrowsMadepg,t1_p12_reboundsTotalpg,t1_p12_turnoverspg,t1_p13_numMinutespg,t1_p13_pointspg,t1_p13_assistspg,t1_p13_blockspg,t1_p13_stealspg,t1_p13_fieldGoalsAttemptedpg,t1_p13_fieldGoalsMadepg,t1_p13_threePointersAttemptedpg,t1_p13_threePointersMadepg,t1_p13_freeThrowsAttemptedpg,t1_p13_freeThrowsMadepg,t1_p13_reboundsTotalpg,t1_p13_turnoverspg,t1_p14_numMinutespg,t1_p14_pointspg,t1_p14_assistspg,t1_p14_blockspg,t1_p14_stealspg,t1_p14_fieldGoalsAttemptedpg,t1_p14_fieldGoalsMadepg,t1_p14_threePointersAttemptedpg,t1_p14_threePointersMadepg,t1_p14_freeThrowsAttemptedpg,t1_p14_freeThrowsMadepg,t1_p14_reboundsTotalpg,t1_p14_turnoverspg
i64,i64,i64,i64,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
22401193,111,126,11,8,2024,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22401194,132,97,15,7,2024,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22401195,116,105,18,32,2024,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22401196,100,115,20,22,2024,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22401197,125,118,28,31,2024,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
28000005,85,95,9,34,1980,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
28000006,98,99,30,14,1980,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
28000007,130,103,2,6,1980,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [24]:
# max_players = None
# max_rows = 0 
# for idx, gameId in enumerate(unique_game_ids["gameId"]):
#     teams = master.filter(pl.col("gameId") == gameId).select(["encodedHomeTeam", "encodedAwayTeam"])
#     rows = box_score_with_per_game.filter((pl.col("gameId") == gameId) & (pl.col("encodedTeam").is_in([teams["encodedHomeTeam"][0], teams["encodedAwayTeam"][0]]))).shape[0]
#     if max_rows < rows:
#         max_players = box_score_with_per_game.filter((pl.col("gameId") == gameId) & (pl.col("encodedTeam").is_in([teams["encodedHomeTeam"][0], teams["encodedAwayTeam"][0]])))
#         max_rows = rows
#         print(max_rows)
#         print(max_players)

In [25]:
joined = box_score_with_per_game.join(
    master.select(["gameId", "encodedHomeTeam", "encodedAwayTeam"]),
    on="gameId",
    how="inner"
)

filtered = joined.filter(
    (pl.col("encodedTeam") == pl.col("encodedHomeTeam")) |
    (pl.col("encodedTeam") == pl.col("encodedAwayTeam"))
).fill_null(0)

player_counts = filtered.group_by("gameId").agg(
    pl.count().alias("numPlayers")
).sort(by="numPlayers", descending=True)

player_counts

/var/folders/x3/q9hsrj516xsbqf99gftzt4vr0000gn/T/ipykernel_44775/3467844820.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("numPlayers")


gameId,numPlayers
i64,u32
22000230,48
22000123,47
20800840,46
22000789,46
22000748,45
…,…
28800843,16
29500512,15
29000999,15


In [26]:
game = games.filter(pl.col("gameId") == 22400985)
game

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season
i64,i64,i64,i64,i64,i32
22400985,144,137,11,24,2024


In [27]:
t0 = filtered.filter((pl.col("gameId") == 22400985) & (pl.col("encodedTeam") == 11)).sort(by="numMinutes", descending=True).limit(15)
t0

firstName,lastName,personId,gameId,playerteamCity,playerteamName,opponentteamCity,opponentteamName,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeam,season,encodedTeampg,numMinutespg,pointspg,assistspg,blockspg,stealspg,fieldGoalsAttemptedpg,fieldGoalsMadepg,threePointersAttemptedpg,threePointersMadepg,freeThrowsAttemptedpg,freeThrowsMadepg,reboundsTotalpg,turnoverspg,encodedHomeTeam,encodedAwayTeam
str,str,i64,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""Jalen""","""Green""",1630224,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",45.18,30.0,13.0,1.0,2.0,26.0,10.0,17.0,5.0,5.0,5.0,7.0,4.0,11,2024,11,32.696049,20.962963,3.481481,0.320988,0.851852,17.444444,7.37037,8.049383,2.876543,4.123457,3.345679,4.567901,2.419753,11,24
"""Dillon""","""Brooks""",1628415,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",40.27,25.0,5.0,1.0,2.0,19.0,8.0,11.0,5.0,4.0,4.0,6.0,0.0,11,2024,11,31.628378,13.358974,1.628205,0.205128,0.769231,11.320513,4.871795,5.910256,2.346154,1.551282,1.269231,3.5,0.935897,11,24
"""Tari""","""Eason""",1631106,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",39.09,21.0,1.0,1.0,0.0,19.0,8.0,7.0,2.0,4.0,3.0,8.0,1.0,11,2024,11,24.718596,9.146667,1.106667,0.666667,1.293333,7.453333,3.626667,2.453333,0.84,1.386667,1.053333,4.826667,0.866667,11,24
"""Jabari""","""Smith Jr.""",1631095,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",38.36,30.0,3.0,0.0,2.0,17.0,11.0,7.0,5.0,4.0,3.0,8.0,4.0,11,2024,11,29.953509,9.184211,0.789474,0.539474,0.328947,7.447368,3.263158,3.644737,1.289474,1.657895,1.368421,5.25,0.802632,11,24
"""Fred""","""VanVleet""",1627832,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",38.28,3.0,5.0,2.0,2.0,6.0,1.0,5.0,1.0,0.0,0.0,7.0,2.0,11,2024,11,34.988305,11.186667,4.36,0.333333,1.24,10.0,3.8,6.053333,2.106667,1.826667,1.48,2.893333,1.133333,11,24
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jeff""","""Green""",201145,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11,2024,11,12.183438,2.123457,0.246914,0.049383,0.074074,1.493827,0.753086,0.975309,0.358025,0.320988,0.259259,0.716049,0.111111,11,24
"""Jack""","""McVeigh""",1629098,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11,2024,11,4.572222,0.608696,0.043478,0.086957,0.0,0.73913,0.217391,0.565217,0.173913,0.0,0.0,0.217391,0.086957,11,24
"""David""","""Roddy""",1631223,22400985,"""Houston""","""Rockets""","""Philadelphia""","""76ers""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11,2024,11,11.37,2.6,0.4,0.2,0.0,2.6,1.0,1.4,0.2,0.8,0.4,1.0,0.2,11,24


In [28]:
t1 = filtered.filter((pl.col("gameId") == 22400985) & (pl.col("encodedTeam") == 24)).sort(by="numMinutes", descending=True).limit(15)
t1

firstName,lastName,personId,gameId,playerteamCity,playerteamName,opponentteamCity,opponentteamName,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeam,season,encodedTeampg,numMinutespg,pointspg,assistspg,blockspg,stealspg,fieldGoalsAttemptedpg,fieldGoalsMadepg,threePointersAttemptedpg,threePointersMadepg,freeThrowsAttemptedpg,freeThrowsMadepg,reboundsTotalpg,turnoverspg,encodedHomeTeam,encodedAwayTeam
str,str,i64,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""Quentin""","""Grimes""",1629656,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",43.01,46.0,4.0,1.0,2.0,27.0,15.0,14.0,8.0,13.0,8.0,13.0,6.0,24,2024,24,33.5125,19.125,3.96875,0.375,1.3125,14.3125,6.71875,6.875,2.5625,4.15625,3.125,4.53125,2.5,11,24
"""Quentin""","""Grimes""",1629656,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",43.01,46.0,4.0,1.0,2.0,27.0,15.0,14.0,8.0,13.0,8.0,13.0,6.0,24,2024,7,22.568298,10.0,2.104167,0.208333,0.666667,7.604167,3.520833,4.1875,1.666667,1.6875,1.291667,3.729167,1.270833,11,24
"""Oshae""","""Brissett""",1629052,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",42.52,11.0,2.0,2.0,0.0,9.0,4.0,4.0,1.0,5.0,2.0,5.0,4.0,24,2024,24,23.425,8.666667,0.666667,0.5,0.666667,6.5,3.166667,3.0,1.0,2.333333,1.333333,3.666667,1.333333,11,24
"""Justin""","""Edwards""",1642348,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",39.38,18.0,4.0,0.0,1.0,15.0,7.0,7.0,4.0,0.0,0.0,3.0,0.0,24,2024,24,26.055682,7.416667,1.15,0.266667,0.75,6.3,2.866667,3.166667,1.15,0.766667,0.533333,2.466667,0.816667,11,24
"""Jared""","""Butler""",1630215,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",37.07,21.0,5.0,1.0,2.0,15.0,7.0,5.0,1.0,6.0,6.0,2.0,1.0,24,2024,24,24.151429,10.354839,4.451613,0.258065,0.967742,8.935484,3.806452,4.129032,1.451613,1.483871,1.290323,2.225806,1.645161,11,24
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Chuma""","""Okeke""",1629643,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",17.19,2.0,1.0,0.0,1.0,2.0,1.0,1.0,0.0,0.0,0.0,3.0,1.0,24,2024,6,12.38,1.0,0.4,0.2,0.0,1.4,0.4,1.2,0.2,0.0,0.0,0.8,0.0,11,24
"""Jeff""","""Dowtin Jr.""",1630288,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",15.52,11.0,2.0,0.0,0.0,8.0,4.0,5.0,3.0,0.0,0.0,3.0,3.0,24,2024,24,14.899512,5.7,1.56,0.24,0.48,4.6,2.24,1.4,0.56,0.9,0.66,1.2,0.3,11,24
"""Alex""","""Reese""",1642024,22400985,"""Philadelphia""","""76ers""","""Houston""","""Rockets""",13.09,6.0,0.0,0.0,0.0,4.0,2.0,4.0,2.0,0.0,0.0,1.0,0.0,24,2024,24,15.15,4.933333,0.266667,0.666667,0.666667,3.533333,1.666667,2.733333,1.0,0.8,0.6,3.066667,0.4,11,24


In [29]:
features = filtered.drop(["playerteamCity",	"playerteamName",	"opponentteamCity",	"opponentteamName", "season", "encodedHomeTeam",	"encodedAwayTeam",])
col_to_move = "encodedTeam"
target_index = 4

# Create new column order
cols = features.columns.copy()
cols.remove(col_to_move)
cols.insert(target_index, col_to_move)

# Reorder DataFrame
features = features.select(cols)
features.filter((pl.col("firstName") == "Stephen") & (pl.col("pointspg") > 29))

firstName,lastName,personId,gameId,encodedTeam,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeampg,numMinutespg,pointspg,assistspg,blockspg,stealspg,fieldGoalsAttemptedpg,fieldGoalsMadepg,threePointersAttemptedpg,threePointersMadepg,freeThrowsAttemptedpg,freeThrowsMadepg,reboundsTotalpg,turnoverspg
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Stephen""","""Curry""",201939,22201230,10,22.0,26.0,7.0,0.0,0.0,15.0,9.0,10.0,5.0,3.0,3.0,5.0,1.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Stephen""","""Curry""",201939,22201211,10,32.0,25.0,6.0,1.0,2.0,14.0,8.0,7.0,3.0,7.0,6.0,7.0,5.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Stephen""","""Curry""",201939,22201187,10,36.0,34.0,6.0,0.0,1.0,25.0,11.0,13.0,6.0,6.0,6.0,5.0,0.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Stephen""","""Curry""",201939,22201175,10,36.0,21.0,4.0,2.0,0.0,28.0,8.0,14.0,2.0,3.0,3.0,3.0,3.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Stephen""","""Curry""",201939,22201158,10,32.0,33.0,5.0,0.0,0.0,21.0,11.0,11.0,7.0,4.0,4.0,2.0,1.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Stephen""","""Curry""",201939,22200068,10,36.0,33.0,9.0,0.0,1.0,22.0,13.0,14.0,7.0,0.0,0.0,7.0,3.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Stephen""","""Curry""",201939,22200055,10,29.0,21.0,8.0,0.0,1.0,17.0,7.0,9.0,4.0,5.0,3.0,7.0,1.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Stephen""","""Curry""",201939,22200042,10,31.0,33.0,2.0,0.0,1.0,22.0,11.0,12.0,7.0,4.0,4.0,5.0,2.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429


In [30]:
master.schema

Schema([('gameId', Int64),
        ('homeScore', Int64),
        ('awayScore', Int64),
        ('encodedHomeTeam', Int64),
        ('encodedAwayTeam', Int64),
        ('season', Int32),
        ('t0_p0_numMinutes', Int32),
        ('t0_p0_points', Int32),
        ('t0_p0_assists', Int32),
        ('t0_p0_blocks', Int32),
        ('t0_p0_steals', Int32),
        ('t0_p0_reboundsTotal', Int32),
        ('t0_p0_turnovers', Int32),
        ('t0_p1_numMinutes', Int32),
        ('t0_p1_points', Int32),
        ('t0_p1_assists', Int32),
        ('t0_p1_blocks', Int32),
        ('t0_p1_steals', Int32),
        ('t0_p1_reboundsTotal', Int32),
        ('t0_p1_turnovers', Int32),
        ('t0_p2_numMinutes', Int32),
        ('t0_p2_points', Int32),
        ('t0_p2_assists', Int32),
        ('t0_p2_blocks', Int32),
        ('t0_p2_steals', Int32),
        ('t0_p2_reboundsTotal', Int32),
        ('t0_p2_turnovers', Int32),
        ('t0_p3_numMinutes', Int32),
        ('t0_p3_points', Int32),
      

In [31]:
features.schema

Schema([('firstName', String),
        ('lastName', String),
        ('personId', Int64),
        ('gameId', Int64),
        ('encodedTeam', Int64),
        ('numMinutes', Float64),
        ('points', Float64),
        ('assists', Float64),
        ('blocks', Float64),
        ('steals', Float64),
        ('fieldGoalsAttempted', Float64),
        ('fieldGoalsMade', Float64),
        ('threePointersAttempted', Float64),
        ('threePointersMade', Float64),
        ('freeThrowsAttempted', Float64),
        ('freeThrowsMade', Float64),
        ('reboundsTotal', Float64),
        ('turnovers', Float64),
        ('encodedTeampg', UInt32),
        ('numMinutespg', Float64),
        ('pointspg', Float64),
        ('assistspg', Float64),
        ('blockspg', Float64),
        ('stealspg', Float64),
        ('fieldGoalsAttemptedpg', Float64),
        ('fieldGoalsMadepg', Float64),
        ('threePointersAttemptedpg', Float64),
        ('threePointersMadepg', Float64),
        ('freeThrowsAtt

In [32]:
features.filter((pl.col("gameId") == 22201230) & (pl.col("encodedTeam" ) == 10)).sort(by="numMinutes", descending=True)

firstName,lastName,personId,gameId,encodedTeam,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,threePointersAttempted,threePointersMade,freeThrowsAttempted,freeThrowsMade,reboundsTotal,turnovers,encodedTeampg,numMinutespg,pointspg,assistspg,blockspg,stealspg,fieldGoalsAttemptedpg,fieldGoalsMadepg,threePointersAttemptedpg,threePointersMadepg,freeThrowsAttemptedpg,freeThrowsMadepg,reboundsTotalpg,turnoverspg
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Moses""","""Moody""",1630541,22201230,10,29.0,25.0,4.0,0.0,0.0,14.0,10.0,7.0,4.0,1.0,1.0,4.0,0.0,10,12.492063,3.822785,0.64557,0.088608,0.227848,2.873418,1.367089,1.708861,0.620253,0.670886,0.468354,1.329114,0.43038
"""Jonathan""","""Kuminga""",1630228,22201230,10,24.0,15.0,6.0,2.0,1.0,9.0,6.0,1.0,1.0,2.0,2.0,7.0,3.0,10,20.328358,9.380282,1.760563,0.43662,0.577465,7.0,3.676056,2.056338,0.760563,1.943662,1.267606,3.253521,1.338028
"""Stephen""","""Curry""",201939,22201230,10,22.0,26.0,7.0,0.0,0.0,15.0,9.0,10.0,5.0,3.0,3.0,5.0,1.0,10,34.125,29.428571,6.285714,0.357143,0.928571,20.232143,9.982143,11.410714,4.875,5.017857,4.589286,6.089286,3.196429
"""Klay""","""Thompson""",202691,22201230,10,21.0,20.0,1.0,0.0,0.0,14.0,7.0,11.0,6.0,0.0,0.0,5.0,3.0,10,32.521739,20.671233,2.232877,0.39726,0.671233,17.150685,7.479452,10.013699,4.123288,1.808219,1.589041,3.917808,1.684932
"""Kevon""","""Looney""",1626172,22201230,10,21.0,4.0,4.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0,8.0,0.0,10,23.365854,7.04878,2.52439,0.609756,0.634146,4.682927,2.95122,0.012195,0.0,1.890244,1.146341,9.268293,0.54878
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jordan""","""Poole""",1629673,22201230,10,17.0,21.0,2.0,0.0,0.0,9.0,7.0,5.0,4.0,4.0,3.0,4.0,2.0,10,29.487805,20.426829,4.5,0.256098,0.768293,15.585366,6.707317,7.768293,2.609756,5.060976,4.402439,2.743902,3.073171
"""JaMychal""","""Green""",203210,22201230,10,11.0,2.0,1.0,1.0,0.0,2.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,10,13.438596,5.382353,0.75,0.338235,0.367647,3.705882,2.0,1.632353,0.617647,0.985294,0.764706,3.014706,0.764706
"""Anthony""","""Lamb""",1630237,22201230,10,10.0,4.0,3.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,3.0,0.0,10,18.806452,6.693548,1.548387,0.322581,0.451613,5.064516,2.387097,3.209677,1.177419,0.967742,0.741935,3.467742,0.870968


In [33]:
# for gameRow in master.iter_rows(named=True):
#     game = gameRow["gameId"]
#     home = gameRow["encodedHomeTeam"]
#     away = gameRow["encodedAwayTeam"]
#     t0 = features.filter((pl.col("gameId") == game) & (pl.col("encodedTeam") == home)).sort(by="numMinutes", descending=True).limit(15)
#     t1 = features.filter((pl.col("gameId") == game) & (pl.col("encodedTeam") == away)).sort(by="numMinutes", descending=True).limit(15)
    

In [34]:

# Step 1: Join features with master to bring in team info
features_with_team_info = features.join(
    master.select(["gameId", "encodedHomeTeam", "encodedAwayTeam"]),
    on="gameId",
    how="inner"
)

# Step 2: Tag each player as Team 0 or 1
features_tagged = features_with_team_info.with_columns([
    pl.when(pl.col("encodedTeam") == pl.col("encodedHomeTeam")).then(0)
     .when(pl.col("encodedTeam") == pl.col("encodedAwayTeam")).then(1)
     .otherwise(None)
     .alias("teamIndex")
]).filter(pl.col("teamIndex").is_not_null())

# Step 3: Rank by numMinutes within each (gameId, teamIndex)
features_ranked = features_tagged.with_columns([
    pl.col("numMinutes").rank("dense", descending=True)
    .over(["gameId", "teamIndex"])
    .alias("playerRank")
])

# Step 4: Keep top 15 players per team
top_players = features_ranked.filter(pl.col("playerRank") <= 15)

# Step 5: Reshape (melt → pivot → wide format)
stat_cols = [
    "numMinutespg", "pointspg", "assistspg", "blockspg", "stealspg",
    "fieldGoalsAttemptedpg", "fieldGoalsMadepg", "threePointersAttemptedpg",
    "threePointersMadepg", "freeThrowsAttemptedpg", "freeThrowsMadepg",
    "reboundsTotalpg", "turnoverspg",
    "numMinutes", "points", "assists", "blocks", "steals",
    "reboundsTotal", "turnovers"
]

melted = top_players.melt(
    id_vars=["gameId", "teamIndex", "playerRank"],
    value_vars=stat_cols
).with_columns([
    (pl.lit("t") + pl.col("teamIndex").cast(pl.Utf8) +
     pl.lit("_p") + (pl.col("playerRank") - 1).cast(pl.Utf8) +
     pl.lit("_") + pl.col("variable")).alias("column_name")
])

wide = melted.pivot(
    values="value",
    index="gameId",
    columns="column_name",
    aggregate_function="first"
)
# Step 6: Join back to master
master_with_features = master.update(wide, on="gameId")

/var/folders/x3/q9hsrj516xsbqf99gftzt4vr0000gn/T/ipykernel_44775/720648036.py:36: DeprecationWarning: `DataFrame.melt` is deprecated. Use `unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  melted = top_players.melt(
/var/folders/x3/q9hsrj516xsbqf99gftzt4vr0000gn/T/ipykernel_44775/720648036.py:45: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  wide = melted.pivot(


In [35]:
master_with_features

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season,t0_p0_numMinutes,t0_p0_points,t0_p0_assists,t0_p0_blocks,t0_p0_steals,t0_p0_reboundsTotal,t0_p0_turnovers,t0_p1_numMinutes,t0_p1_points,t0_p1_assists,t0_p1_blocks,t0_p1_steals,t0_p1_reboundsTotal,t0_p1_turnovers,t0_p2_numMinutes,t0_p2_points,t0_p2_assists,t0_p2_blocks,t0_p2_steals,t0_p2_reboundsTotal,t0_p2_turnovers,t0_p3_numMinutes,t0_p3_points,t0_p3_assists,t0_p3_blocks,t0_p3_steals,t0_p3_reboundsTotal,t0_p3_turnovers,t0_p4_numMinutes,t0_p4_points,t0_p4_assists,…,t1_p12_assistspg,t1_p12_blockspg,t1_p12_stealspg,t1_p12_fieldGoalsAttemptedpg,t1_p12_fieldGoalsMadepg,t1_p12_threePointersAttemptedpg,t1_p12_threePointersMadepg,t1_p12_freeThrowsAttemptedpg,t1_p12_freeThrowsMadepg,t1_p12_reboundsTotalpg,t1_p12_turnoverspg,t1_p13_numMinutespg,t1_p13_pointspg,t1_p13_assistspg,t1_p13_blockspg,t1_p13_stealspg,t1_p13_fieldGoalsAttemptedpg,t1_p13_fieldGoalsMadepg,t1_p13_threePointersAttemptedpg,t1_p13_threePointersMadepg,t1_p13_freeThrowsAttemptedpg,t1_p13_freeThrowsMadepg,t1_p13_reboundsTotalpg,t1_p13_turnoverspg,t1_p14_numMinutespg,t1_p14_pointspg,t1_p14_assistspg,t1_p14_blockspg,t1_p14_stealspg,t1_p14_fieldGoalsAttemptedpg,t1_p14_fieldGoalsMadepg,t1_p14_threePointersAttemptedpg,t1_p14_threePointersMadepg,t1_p14_freeThrowsAttemptedpg,t1_p14_freeThrowsMadepg,t1_p14_reboundsTotalpg,t1_p14_turnoverspg
i64,i64,i64,i64,i64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
22401193,111,126,11,8,2024,27.14,15.0,3.0,0.0,1.0,0.0,3.0,26.13,15.0,6.0,1.0,1.0,6.0,1.0,23.34,14.0,4.0,1.0,0.0,5.0,0.0,22.12,8.0,1.0,0.0,0.0,1.0,0.0,20.39,2.0,2.0,…,0.298701,0.012987,0.090909,0.753247,0.272727,0.337662,0.090909,0.12987,0.090909,0.649351,0.194805,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22401194,132,97,15,7,2024,37.16,23.0,7.0,0.0,3.0,2.0,0.0,35.58,31.0,3.0,0.0,0.0,6.0,3.0,34.17,5.0,3.0,0.0,0.0,4.0,2.0,30.46,25.0,3.0,0.0,1.0,12.0,1.0,28.05,12.0,5.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22401195,116,105,18,32,2024,40.15,43.0,5.0,1.0,3.0,6.0,2.0,38.28,19.0,0.0,4.0,1.0,18.0,0.0,32.58,10.0,5.0,1.0,0.0,10.0,4.0,32.4,9.0,7.0,1.0,2.0,5.0,1.0,28.18,16.0,2.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22401196,100,115,20,22,2024,39.23,15.0,2.0,1.0,2.0,6.0,2.0,39.1,20.0,0.0,1.0,0.0,6.0,0.0,33.37,14.0,4.0,0.0,2.0,3.0,6.0,33.35,17.0,6.0,0.0,0.0,5.0,0.0,33.16,10.0,5.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22401197,125,118,28,31,2024,37.16,20.0,6.0,0.0,0.0,8.0,4.0,34.45,14.0,1.0,1.0,0.0,6.0,0.0,34.12,18.0,1.0,0.0,3.0,5.0,0.0,33.51,15.0,6.0,0.0,4.0,7.0,1.0,27.12,23.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
28000005,85,95,9,34,1980,40.0,4.0,6.0,0.0,0.0,3.0,0.0,36.0,12.0,4.0,0.0,0.0,13.0,0.0,34.0,24.0,2.0,0.0,0.0,3.0,0.0,31.0,16.0,2.0,0.0,0.0,5.0,0.0,25.0,14.0,1.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28000006,98,99,30,14,1980,43.0,24.0,5.0,2.0,3.0,1.0,1.0,38.0,10.0,2.0,0.0,0.0,12.0,3.0,36.0,19.0,6.0,0.0,3.0,5.0,9.0,32.0,12.0,3.0,0.0,0.0,7.0,2.0,31.0,13.0,2.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [36]:
master_with_features

# Assume df is your Polars DataFrame
all_cols = master_with_features.columns

# Split columns into two groups
num_minutes_cols = [col for col in all_cols if col.endswith("numMinutes")]
other_cols = [col for col in all_cols if not col.endswith("numMinutes")]

# Reorder
master_with_features = master_with_features.select(other_cols + num_minutes_cols)
master_with_features

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season,t0_p0_points,t0_p0_assists,t0_p0_blocks,t0_p0_steals,t0_p0_reboundsTotal,t0_p0_turnovers,t0_p1_points,t0_p1_assists,t0_p1_blocks,t0_p1_steals,t0_p1_reboundsTotal,t0_p1_turnovers,t0_p2_points,t0_p2_assists,t0_p2_blocks,t0_p2_steals,t0_p2_reboundsTotal,t0_p2_turnovers,t0_p3_points,t0_p3_assists,t0_p3_blocks,t0_p3_steals,t0_p3_reboundsTotal,t0_p3_turnovers,t0_p4_points,t0_p4_assists,t0_p4_blocks,t0_p4_steals,t0_p4_reboundsTotal,t0_p4_turnovers,t0_p5_points,…,t1_p14_fieldGoalsMadepg,t1_p14_threePointersAttemptedpg,t1_p14_threePointersMadepg,t1_p14_freeThrowsAttemptedpg,t1_p14_freeThrowsMadepg,t1_p14_reboundsTotalpg,t1_p14_turnoverspg,t0_p0_numMinutes,t0_p1_numMinutes,t0_p2_numMinutes,t0_p3_numMinutes,t0_p4_numMinutes,t0_p5_numMinutes,t0_p6_numMinutes,t0_p7_numMinutes,t0_p8_numMinutes,t0_p9_numMinutes,t0_p10_numMinutes,t0_p11_numMinutes,t0_p12_numMinutes,t0_p13_numMinutes,t0_p14_numMinutes,t1_p0_numMinutes,t1_p1_numMinutes,t1_p2_numMinutes,t1_p3_numMinutes,t1_p4_numMinutes,t1_p5_numMinutes,t1_p6_numMinutes,t1_p7_numMinutes,t1_p8_numMinutes,t1_p9_numMinutes,t1_p10_numMinutes,t1_p11_numMinutes,t1_p12_numMinutes,t1_p13_numMinutes,t1_p14_numMinutes
i64,i64,i64,i64,i64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
22401193,111,126,11,8,2024,15.0,3.0,0.0,1.0,0.0,3.0,15.0,6.0,1.0,1.0,6.0,1.0,14.0,4.0,1.0,0.0,5.0,0.0,8.0,1.0,0.0,0.0,1.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,8.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.14,26.13,23.34,22.12,20.39,17.56,17.13,12.0,11.31,9.23,4.08,0.0,0.0,0.0,0.0,31.23,30.46,29.29,26.23,24.47,22.39,19.28,17.08,10.52,9.52,6.22,4.29,0.0,0.0,0.0
22401194,132,97,15,7,2024,23.0,7.0,0.0,3.0,2.0,0.0,31.0,3.0,0.0,0.0,6.0,3.0,5.0,3.0,0.0,0.0,4.0,2.0,25.0,3.0,0.0,1.0,12.0,1.0,12.0,5.0,0.0,1.0,5.0,1.0,22.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.16,35.58,34.17,30.46,28.05,27.51,24.22,21.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.18,35.51,33.45,27.33,26.16,21.28,18.44,15.48,14.17,7.0,0.0,0.0,0.0,0.0,0.0
22401195,116,105,18,32,2024,43.0,5.0,1.0,3.0,6.0,2.0,19.0,0.0,4.0,1.0,18.0,0.0,10.0,5.0,1.0,0.0,10.0,4.0,9.0,7.0,1.0,2.0,5.0,1.0,16.0,2.0,0.0,0.0,3.0,1.0,8.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.15,38.28,32.58,32.4,28.18,25.07,24.34,14.58,2.42,0.0,0.0,0.0,0.0,0.0,0.0,48.0,39.23,36.46,29.04,28.31,27.33,24.35,6.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22401196,100,115,20,22,2024,15.0,2.0,1.0,2.0,6.0,2.0,20.0,0.0,1.0,0.0,6.0,0.0,14.0,4.0,0.0,2.0,3.0,6.0,17.0,6.0,0.0,0.0,5.0,0.0,10.0,5.0,0.0,1.0,16.0,2.0,18.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.23,39.1,33.37,33.35,33.16,28.01,17.31,15.27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.33,37.06,36.3,36.08,34.48,21.55,17.5,14.53,2.16,0.0,0.0,0.0,0.0,0.0,0.0
22401197,125,118,28,31,2024,20.0,6.0,0.0,0.0,8.0,4.0,14.0,1.0,1.0,0.0,6.0,0.0,18.0,1.0,0.0,3.0,5.0,0.0,15.0,6.0,0.0,4.0,7.0,1.0,23.0,0.0,1.0,0.0,9.0,2.0,8.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.16,34.45,34.12,33.51,27.12,19.31,18.58,16.44,14.03,3.28,0.0,0.0,0.0,0.0,0.0,42.21,37.31,36.34,36.13,34.55,29.16,21.18,1.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
28000005,85,95,9,34,1980,4.0,6.0,0.0,0.0,3.0,0.0,12.0,4.0,0.0,0.0,13.0,0.0,24.0,2.0,0.0,0.0,3.0,0.0,16.0,2.0,0.0,0.0,5.0,0.0,14.0,1.0,0.0,0.0,8.0,0.0,3.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,36.0,34.0,31.0,25.0,18.0,14.0,13.0,12.0,10.0,7.0,0.0,0.0,0.0,0.0,42.0,35.0,33.0,31.0,27.0,23.0,22.0,17.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
28000006,98,99,30,14,1980,24.0,5.0,2.0,3.0,1.0,1.0,10.0,2.0,0.0,0.0,12.0,3.0,19.0,6.0,0.0,3.0,5.0,9.0,12.0,3.0,0.0,0.0,7.0,2.0,13.0,2.0,0.0,2.0,2.0,0.0,11.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.0,38.0,36.0,32.0,31.0,22.0,21.0,7.0,6.0,4.0,0.0,0.0,0.0,0.0,0.0,42.0,41.0,39.0,36.0,34.0,25.0,19.0,3.0,1.0,0.0,0.0

In [37]:
master_with_features.write_csv("../csv/masterGame.csv")

In [38]:
null_counts = master_with_features.select([
    pl.col(col).is_null().sum().alias(col)
    for col in master_with_features.columns
])
null_counts

gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season,t0_p0_points,t0_p0_assists,t0_p0_blocks,t0_p0_steals,t0_p0_reboundsTotal,t0_p0_turnovers,t0_p1_points,t0_p1_assists,t0_p1_blocks,t0_p1_steals,t0_p1_reboundsTotal,t0_p1_turnovers,t0_p2_points,t0_p2_assists,t0_p2_blocks,t0_p2_steals,t0_p2_reboundsTotal,t0_p2_turnovers,t0_p3_points,t0_p3_assists,t0_p3_blocks,t0_p3_steals,t0_p3_reboundsTotal,t0_p3_turnovers,t0_p4_points,t0_p4_assists,t0_p4_blocks,t0_p4_steals,t0_p4_reboundsTotal,t0_p4_turnovers,t0_p5_points,…,t1_p14_fieldGoalsMadepg,t1_p14_threePointersAttemptedpg,t1_p14_threePointersMadepg,t1_p14_freeThrowsAttemptedpg,t1_p14_freeThrowsMadepg,t1_p14_reboundsTotalpg,t1_p14_turnoverspg,t0_p0_numMinutes,t0_p1_numMinutes,t0_p2_numMinutes,t0_p3_numMinutes,t0_p4_numMinutes,t0_p5_numMinutes,t0_p6_numMinutes,t0_p7_numMinutes,t0_p8_numMinutes,t0_p9_numMinutes,t0_p10_numMinutes,t0_p11_numMinutes,t0_p12_numMinutes,t0_p13_numMinutes,t0_p14_numMinutes,t1_p0_numMinutes,t1_p1_numMinutes,t1_p2_numMinutes,t1_p3_numMinutes,t1_p4_numMinutes,t1_p5_numMinutes,t1_p6_numMinutes,t1_p7_numMinutes,t1_p8_numMinutes,t1_p9_numMinutes,t1_p10_numMinutes,t1_p11_numMinutes,t1_p12_numMinutes,t1_p13_numMinutes,t1_p14_numMinutes
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [39]:


player_stats_raw = player_stats_raw.with_columns(
    pl.col("playerteamCity").map_elements(lambda x: mapping_dict.get(x, -1), return_dtype=pl.Int64).alias("encodedTeam")
)

player_stats_raw.write_csv("../csv/PlayerStatistics.csv")